# Session 14 — dense linear FFN, then MoE

Train the dense GPT, then keep training four ways from the same checkpoint: the original linear block, a **copy** upcycle, a **drop** upcycle (`r = 0.5`), and a **partition** (shared expert + routed halves). Numbers below are read from `results.json`. The step log is `logs/train.jsonl` (2,500 lines).

In [1]:
import json
from pathlib import Path

root = Path('.').resolve()
d = json.loads((root / 'results.json').read_text())
print(f"device {d['device']}  torch {d['torch']}  dense params {d['dense_params']:,}")
print('router check', d['router_check'])
print(f"{'phase':16} {'start':>7} {'end':>7} {'mean50':>7} {'tok/s':>8} {'params':>10}")
for ph in d['phases']:
    print(f"{ph['phase']:16} {ph['start_loss']:7.3f} {ph['end_loss']:7.3f} {ph['mean_last_50']:7.3f} {ph['tok_s']:8.0f} {ph['params']:10,}")
print('log lines', sum(1 for _ in open(root / 'logs' / 'train.jsonl')))


device mps  torch 2.13.0  dense params 1,323,776
router check {'softmax_0': 0.32, 'softmax_5': 0.262, 'weight_0': 0.55, 'weight_5': 0.45, 'kept_experts': [0, 5]}
phase              start     end  mean50    tok/s     params
dense              8.345   4.347   4.387    53931  1,323,776
dense_continue     4.368   3.557   3.904    56671  1,323,776
moe_drop           4.486   3.497   3.840     7687  5,015,808
moe_copy           4.368   3.518   3.857     6358  5,015,808
moe_partition      4.378   3.531   3.883     6970  3,437,312
log lines 2500


![loss](figures/loss_phases.png)

The dashed line is the upcycle. After it, dense continuation, drop, copy, and partition all keep falling. Drop starts a little higher because half the hidden units are new. Copy matches the dense loss at the first MoE step. Partition is the arm whose routed matrices shrink from 512 to 256.

![load](figures/expert_load.png)

Even share is 2/8 = 0.25. No expert is dead on the last step of copy, drop, or partition.

A second log, `logs/extras.jsonl`, checks the lecture failure modes. With 8 experts, the bias (γ = 0.001) leaves the loss alone and cuts max load violation to 0.211, against 0.558 with no balancer. Cloning those 8 into 32 identical experts and using hard top-k leaves 8 experts dead at the last step. Probabilistic selection for the first 125 steps keeps the dead count at 0 until the switch back to hard top-k. Redrawing half of each clone leaves 0 dead at the end.

![balancers](figures/balancer_loss.png)

![dead](figures/grow_dead.png)